In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


command_result_schema = ArrayType(
    StructType(
        [
            StructField("worker", StringType(), nullable=False),
            StructField("command", StringType(), nullable=False),
            StructField("exit_code", IntegerType(), nullable=False),
            StructField("output", StringType(), nullable=False),
        ]
    )
)


@F.udf(returnType=command_result_schema)
def start_tunnel_and_run_commands(commands):
    import os
    import socket
    import subprocess

    worker = socket.gethostname()
    results = []

    # Keep every dependency inside the UDF so Spark can execute it
    # independently on the worker.
    CONTROL_SOCKET = "/dev/shm/databricks-ssh"
    SSH_HOST = "databricks@pub.worldb.dedyn.io"
    SSH_PORT = 8889
    SSH_KEY = (
        "/Workspace/Users/rogermm@gmail.com/"
        ".ssh/id_ed25519_databricks_free"
    )

    tools_host = "172.18.0.8"
    host_ip = "localhost"

    ssh_base = (
        f"ssh "
        f"-S {CONTROL_SOCKET} "
        f"-p {SSH_PORT} "
        f"-i {SSH_KEY} "
        f"-o BatchMode=yes "
        f"-o StrictHostKeyChecking=accept-new "
        f"-o UserKnownHostsFile=/tmp/known_hosts "
    )

    check_command = (
        f"{ssh_base}"
        f"-O check "
        f"{SSH_HOST}"
    )

    start_command = (
        f"ssh "
        f"-M "
        f"-S {CONTROL_SOCKET} "
        f"-o ControlPersist=yes "
        f"-o ServerAliveInterval=30 "
        f"-o ServerAliveCountMax=3 "
        f"-o ExitOnForwardFailure=yes "
        f"-o StrictHostKeyChecking=accept-new "
        f"-o UserKnownHostsFile=/tmp/known_hosts "
        f"-o BatchMode=yes "
        f"-p {SSH_PORT} "
        f"-i {SSH_KEY} "
        f"-L 127.0.0.1:9092:kafka-4:9092 "
        f"-L 127.0.0.1:8080:traefik:80 "
        f"-L {host_ip}:8888:{tools_host}:8888 "
        f"-fNT "
        f"{SSH_HOST}"
    )

    def run_command(command):
        result = subprocess.run(
            command,
            shell=True,
            text=True,
            capture_output=True,
            check=False,
        )

        output = "\n".join(
            value.strip()
            for value in (result.stdout, result.stderr)
            if value and value.strip()
        )

        results.append(
            (
                worker,
                command,
                result.returncode,
                output,
            )
        )

        return result

    try:
        # Check whether the tunnel already exists on this worker.
        check_result = subprocess.run(
            check_command,
            shell=True,
            text=True,
            capture_output=True,
            check=False,
        )

        if check_result.returncode != 0:
            # Remove any abandoned multiplexing socket.
            for socket_path in (
                CONTROL_SOCKET,
                f"{CONTROL_SOCKET}.*",
            ):
                subprocess.run(
                    f"rm -f {socket_path}",
                    shell=True,
                    text=True,
                    capture_output=True,
                    check=False,
                )

            # Start the tunnel.
            start_result = run_command(start_command)

            # Stop here if the SSH tunnel could not be created.
            if start_result.returncode != 0:
                return results

        else:
            output = "\n".join(
                value.strip()
                for value in (
                    check_result.stdout,
                    check_result.stderr,
                )
                if value and value.strip()
            )

            results.append(
                (
                    worker,
                    check_command,
                    check_result.returncode,
                    output or "SSH tunnel is already running.",
                )
            )

        # Run the supplied commands after starting the tunnel.
        for command in commands or []:
            run_command(command)

        return results

    except Exception as error:
        results.append(
            (
                worker,
                "start_tunnel_and_run_commands",
                -1,
                f"{type(error).__name__}: {error}",
            )
        )

        return results

In [0]:
def run_worker_commands(commands):
    commands_column = F.array(
        *[F.lit(command) for command in commands]
    )

    return (
        spark.range(1)
        .repartition(1)
        .select(
            F.explode(
                start_tunnel_and_run_commands(commands_column)
            ).alias("result")
        )
        .select(
            F.col("result.worker").alias("worker"),
            F.col("result.command").alias("command"),
            F.col("result.exit_code").alias("exit_code"),
            F.col("result.output").alias("output"),
        )
    )

In [0]:
commands = [
    "hostname",
    "whoami",
    "ps -ef | grep 'ssh'",
    #'nc -zv localhost 8888 2>&1 | grep -v "Warning"',
    "curl -v http://localhost:8888/",
]

df = run_worker_commands(commands)

display(df)

In [0]:
commands = []
df = run_worker_commands(commands)
display(df)

In [0]:
from pyspark.sql import functions as F

empty_commands = F.expr("CAST(array() AS ARRAY<STRING>)")
commands = F.array(F.lit("nc -v localhost 8888"))

start_tunnel_df = (
    spark.range(1, numPartitions=1)
    .select(
        start_tunnel_and_run_commands(commands).alias("result")
    )
)

# Forces the UDF to run on the Spark worker.
df = start_tunnel_df.show(truncate=False)
display(df)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def add_ssh_tunnel_column(
    df: DataFrame,
    column_name: str = "ssh_tunnel",
) -> DataFrame:
    empty_commands = F.array().cast("array<string>")

    return df.withColumn(
        column_name,
        start_tunnel_and_run_commands(empty_commands),
    )

In [0]:
df = spark.range(1, numPartitions=1)

In [0]:
display(df)

In [0]:
df_with_tunnel = add_ssh_tunnel_column(df)

display(df_with_tunnel)